# Loading

In [1]:
import pandas as pd
import geopandas as gpd

# Raw suburban election results (already cleaned: registered, voters_cast, turnout_pct)
suburban_election = pd.read_csv('suburban_cook_election_2022_general_1.csv') 
precinct_lookup = gpd.read_file('Precinct.geojson')[['precinctid', 'Township_Name']].copy()

In [2]:
print(suburban_election.shape)   # expect (1430, 4)
print(precinct_lookup.shape)

(1430, 4)
(1430, 2)


# Analysis

In [3]:
suburban_election['precinct_id'] = suburban_election['precinct_id'].astype(str)
precinct_lookup['precinctid'] = precinct_lookup['precinctid'].astype(str)

merged = suburban_election.merge(precinct_lookup, left_on='precinct_id', right_on='precinctid', how='left')

# Sanity check: how many precincts failed to match a township?
print(merged['Township_Name'].isna().sum())  # NOT expected to be 0 — check this number, investigate if large
print(merged.shape)  # should stay (1430, ...) — how='left' keeps all election rows even if unmatched


133
(1430, 6)


In [4]:
print(merged[merged['Township_Name'].isna()]['precinct_id'].tolist())

['9903005', '7100055', '7100056', '7100057', '7100058', '7100059', '7100061', '7200067', '7200068', '7200069', '7200070', '7200071', '7200072', '7200074', '7200075', '7200077', '7200079', '7200080', '7300012', '9800032', '7400046', '7400047', '7400048', '7400049', '7400050', '7400051', '7503006', '7503007', '7506007', '7507006', '7509006', '7600044', '7900071', '7900072', '7900073', '7900074', '7900076', '7900078', '8000085', '8000086', '8000087', '8000089', '8000090', '8000091', '8100040', '8200063', '8200064', '8300059', '8300060', '8300061', '8300062', '8400016', '8500033', '8500034', '8500035', '8600063', '8600064', '8600065', '8600066', '8600067', '8600068', '8600070', '8600071', '8600073', '8600074', '8600075', '8700066', '8700067', '8700068', '8700069', '8800036', '8800037', '8800038', '8900090', '8900091', '8900092', '8900093', '8900094', '8900095', '8900096', '8900097', '8900099', '8900100', '8900101', '8900102', '9000048', '9000049', '9000050', '9100008', '9200013', '9300070'

In [5]:
unmatched = merged[merged['Township_Name'].isna()]['precinct_id']
print(unmatched.astype(str).str[:2].value_counts())               

precinct_id
95    14
97    13
89    12
86    11
72    11
96     9
93     7
74     6
79     6
80     6
71     6
75     5
83     4
87     4
88     3
90     3
85     3
82     2
91     1
92     1
99     1
81     1
76     1
98     1
73     1
84     1
Name: count, dtype: int64


In [7]:
precinct = gpd.read_file('Precinct.geojson')
print(precinct[['Township_Name', 'Township_Number']].drop_duplicates().sort_values('Township_Number'))

     Township_Name Township_Number
0       BARRINGTON              70
10           BLOOM              71
60          BREMEN              72
129        CALUMET              73
140      ELK GROVE              74
186       EVANSTON              75
231        HANOVER              76
242         LEMONT              77
288         LEYDEN              78
319          LYONS              79
406          MAINE              80
490      NEW TRIER              81
529          NILES              82
590     NORTHFIELD              83
649   NORWOOD PARK              84
663       OAK PARK              85
693         ORLAND              86
757       PALATINE              87
822          PALOS              88
856        PROVISO              89
946           RICH              90
992   RIVER FOREST              91
994      RIVERSIDE              92
1000    SCHAUMBURG              93
1056      STICKNEY              94
1099      THORNTON              95
1203      WHEELING              96
1286         WORTH  

In [8]:
unmatched_rows = merged[merged['Township_Name'].isna()]
print(unmatched_rows['registered'].sum())
print(merged['registered'].sum())
print(unmatched_rows['registered'].sum() / merged['registered'].sum() * 100)

147675
1648706
8.957024478591089


# Townships

In [9]:
township_election = merged.groupby('Township_Name')[['registered', 'voters_cast']].sum().reset_index()
township_election['turnout'] = (township_election['voters_cast'] / township_election['registered']) * 100

print(township_election.shape)   # expect (30, 4) — one row per township
print(township_election.head(10))
print(township_election['turnout'].describe())

(30, 4)
  Township_Name  registered  voters_cast    turnout
0    BARRINGTON       12958         6943  53.580800
1        BERWYN       30407        10444  34.347354
2         BLOOM       52422        19715  37.608256
3        BREMEN       59917        23686  39.531352
4       CALUMET       10574         3526  33.345943
5        CICERO       32174         8165  25.377634
6     ELK GROVE       48436        22158  45.746965
7      EVANSTON       45970        25752  56.019143
8       HANOVER       55807        20215  36.223054
9        LEMONT       17098         9444  55.234530
count    30.000000
mean     45.600426
std       9.026595
min      25.377634
25%      38.089030
50%      45.378217
75%      53.057668
max      59.658722
Name: turnout, dtype: float64


In [10]:
township_election.to_csv('suburban_election_clean.csv', index=False)
print('saved:', township_election.shape)

saved: (30, 4)
